# Projekt 4: Transformacja 3D (Vector Field Flow)

**Autorzy:** Jakub Szewczyk, Miłosz Andruczyk

Celem projektu jest wytrenowanie pól wektorowych transformujących reprezentację punktową (point cloud) obiektów wejściowych w docelową reprezentację czajnika (teapot). Model realizuje płynne przejście między siatką obiektu wejściowego a siatką czajnika.

W ramach eksperymentów wykorzystano architekturę opartą na operacjach grafowych z knn (Dynamic Graph CNN / DGCNN), która na podstawie cech grafu przewiduje przemieszczenia punktów (displacement).

## 1. Opis i założenia zadania

Zadanie polegało na:
1. Wytrenowaniu 3 oddzielnych pól wektorowych dla 3 różnych obiektów (`bunny`, `dragon_small`, `armadillo_small`), uczących się przekształcenia tych obiektów w `teapot`.
2. Sprawdzeniu, czy tak wytrenowane modele są w stanie poprawnie przetransformować wcześniej niewidziany obiekt testowy (`asian_dragon_really_small`).
3. Zastosowaniu odpowiedniej augmentacji (losowa rotacja i skalowanie) podczas uczenia, by uniezależnić model od układu współrzędnych i początkowego rozmiaru.
4. Ocenie wyników za pomocą metryk: Indeksu Jaccarda (IoU), Współczynnika Dice'a oraz odległości Chamfera.

## 2. Instrukcja uruchomienia

Aby odtworzyć wyniki projektu, należy postępować zgodnie z poniższymi krokami:

### Wymagania (Dependencies)
Projekt wymaga środowiska z zainstalowanymi bibliotekami:
```bash
pip install torch numpy scipy trimesh pandas tqdm
```

### Struktura katalogów
Modele w formacie `.obj` lub `.ply` powinny znaleźć się w katalogu określonym w pliku `config.py` (domyślnie `data/`).
- `data/bunny.obj`
- `data/dragon_small.obj`
- `data/armadillo_small.obj`
- `data/asian_dragon_really_small.obj`
- `data/teapot.obj`

### Uruchomienie pipeline'u
Aby rozpocząć proces treningowy oraz ewaluację, wystarczy wywołać główną funkcję pipeline'u z pliku `services.py`:

```python
from services import run_pipeline
run_pipeline()
```
W wyniku wykonania skryptu, w folderze `output/50_0.001_8192/` wygenerowane zostaną: plik z wynikami `evaluation_results.csv`, modele `.pth` w podfolderze `saved_models` oraz chmury punktów reprezentujące transformację (kroki 0.0, 0.5, 1.0) w folderze `samples`.

## 3. Architektury modelu

Zaimplementowany model `VectorFieldDGCNN` bazuje na architekturze **DGCNN (Dynamic Graph CNN)**. Klasyczne sieci przetwarzające chmury punktów (np. PointNet) traktują każdy punkt niezależnie, tracąc informacje o lokalnej topologii geometrycznej. DGCNN rozwiązuje ten problem poprzez wprowadzenie operacji **EdgeConv**.

### Główne komponenty sieci:
1. **Dynamiczny Graf KNN (`knn`):** Dla każdej warstwy sieć buduje graf najbliższych sąsiadów ($k=20$) w oparciu o odległości euklidesowe punktów w przestrzeni cech (a nie tylko w początkowej przestrzeni geometrycznej XYZ). Oznacza to, że graf zmienia się dynamicznie z warstwy na warstwę.
2. **Ekstrakcja cech relacyjnych (`get_graph_feature`):** Dla każdego punktu $x_i$ i jego sąsiada $x_j$, sieć konstruuje cechę krawędzi jako parę: $(x_i, x_j - x_i)$. Taka reprezentacja łączy globalną pozycję punktu z jego lokalną strukturą otoczenia (wektory przesunięć do sąsiadów).
3. **Bloki Konwolucyjne:** Cechy krawędzi są przetwarzane przez wielowarstwowe konwolucje 1D (`nn.Conv1d`) sprzężone z warstwami normalizacji (`nn.BatchNorm1d`) oraz funkcją aktywacji `LeakyReLU(negative_slope=0.2)`. Po każdej konwolucji stosowana jest operacja `max(dim=-1)` agregująca cechy z otoczenia sąsiadów.
4. **Agregacja globalna i lokalna:** Cechy z poszczególnych poziomów sieci są łączone (konkatenacja wektorów cech lokalnych oraz globalnego deskryptora kształtu uzyskanego przez MaxPooling całej chmury).
5. **Dekoder Przemieszczeń (Displacement Decoder):** Połączony wektor trafia do warstw w pełni połączonych (zrealizowanych jako `Conv1d` o jądrze 1). Ostatnia warstwa (`nn.Conv1d(256, 3, 1)`) nie posiada funkcji aktywacji ani normalizacji i zwraca tensor o wymiarach `(B, 3, N)`, reprezentujący trójwymiarowy wektor przesunięcia $\Delta x$ dla każdego punktu. Ostateczna pozycja punktu w przepływie to $x_{pred} = x_{source} + t \cdot \Delta x$.

---

## 4. Funkcja straty: Odległość Chamfera

W zadaniach generowania lub transformacji chmur punktów nie można zastosować klasycznych funkcji straty, takich jak błąd średniokwadratowy (MSE) liczony punkt-do-punktu. Wynika to z faktu, że chmura punktów jest zbiorem **nieuporządkowanym**. Podczas próbkowania powierzchni docelowej siatki (`teapot.obj`), punkty są generowane losowo i ich indeksy nie odpowiadają punktom z modelu wejściowego.

Z tego powodu w pliku `services.py` wdrożono **symetryczną odległość Chamfera (Chamfer Distance Loss)**. Miara ta bada odległość każdego punktu z jednej chmury do najbliższego punktu w drugiej chmurze.

### Matematyczna definicja:
Dla chmury punktów predykcji $P_{pred}$ oraz chmury docelowej $P_{target}$ funkcja straty zdefiniowana jest jako:

$$\mathcal{L}_{Chamfer}(P_{pred}, P_{target}) = \frac{1}{|P_{pred}|} \sum_{x \in P_{pred}} \min_{y \in P_{target}} \|x - y\|^2 + \frac{1}{|P_{target}|} \sum_{y \in P_{target}} \min_{x \in P_{pred}} \|x - y\|^2$$

### Opis składników:
* **Pierwszy składnik (Forward Loss):** Dla każdego punktu wygenerowanego przez model ($x$), szuka najbliższego punktu w prawdziwym czajniku ($y$) i liczy kwadrat odległości. Gwarantuje to, że wygenerowane punkty leżą blisko powierzchni docelowej.
* **Drugi składnik (Backward Loss):** Dla każdego punktu rzeczywistego czajnika ($y$), szuka najbliższego punktu w chmurze wygenerowanej ($x$). Ten składnik zapobiega sytuacji, w której model zapada całą chmurę do jednego punktu (*mode collapse*) i wymusza pokrycie całego docelowego kształtu.

## 5. Parametry modelu i treningu

Najważniejsze hiperparametry zaczerpnięte z `config.py`:
- **Liczba epok:** 50
- **Rozmiar batcha:** 2
- **Learning rate:** 0.001
- **Liczba próbkowanych punktów:** 8192

Do ekstrakcji cech wybrano sieć `VectorFieldDGCNN` z operacją KNN ($k=20$), wdrożoną w module `models.py`.

## 6. Wyniki metryczne

Ewaluacja została przeprowadzona na danych testowych i zebrana w pliku `evaluation_results.csv`. Pod uwagę wzięto odległość Chamfera (im mniejsza, tym lepsza) oraz współczynniki IoU i Dice (im wyższe, tym lepsze).

| Metoda | IoU | Dice | Chamfer |
|:---|:---:|:---:|:---:|
| bunny_flow | 0.2637 | 0.4174 | 0.0026 |
| dragon_small_flow | 0.1837 | 0.3104 | 0.0049 |
| armadillo_small_flow | 0.2298 | 0.3737 | 0.0026 |
| bunny_flow_asian_dragon_really_small | 0.1542 | 0.2672 | 0.0087 |
| dragon_small_flow_asian_dragon_really_small | 0.1908 | 0.3205 | 0.0040 |
| armadillo_small_flow_asian_dragon_really_small | 0.1994 | 0.3324 | 0.0043 |

## 7. Wizualizacja wyników - kroki transformacji (Flow)

Poniżej znajdują się wizualizacje przekształceń na poszczególnych etapach przejścia ($t=0.0$, $t=0.5$, $t=1.0$). Zostały one wygenerowane i zapisane do plików `.obj` podczas ewaluacji.

### Model 1: Bunny $\rightarrow$ Teapot
<div style="display: flex; justify-content: space-between; text-align: center;">
    <div>
        <img src="output\200_0.001_4096\photos\b_0.0.png" alt="Bunny Krok 0.0" width="250"/>
        <p><i>Krok 0.0 (Oryginał)</i></p>
    </div>
    <div>
        <img src="output\200_0.001_4096\photos\b_0.5.png" alt="Bunny Krok 0.5" width="250"/>
        <p><i>Krok 0.5 (Faza przejściowa)</i></p>
    </div>
    <div>
        <img src="output\200_0.001_4096\photos\b_1.0.png" alt="Bunny Krok 1.0" width="250"/>
        <p><i>Krok 1.0 (Teapot)</i></p>
    </div>
</div>

### Model 2: Armadillo $\rightarrow$ Teapot
<div style="display: flex; justify-content: space-between; text-align: center;">
    <div>
        <img src="output\200_0.001_4096\photos\asfs_0.0.png" alt="Armadillo Krok 0.0" width="250"/>
        <p><i>Krok 0.0 (Oryginał)</i></p>
    </div>
    <div>
        <img src="output\200_0.001_4096\photos\asfs_0.5.png" alt="Armadillo Krok 0.5" width="250"/>
        <p><i>Krok 0.5 (Faza przejściowa)</i></p>
    </div>
    <div>
        <img src="output\200_0.001_4096\photos\asfs_1.0.png" alt="Armadillo Krok 1.0" width="250"/>
        <p><i>Krok 1.0 (Teapot)</i></p>
    </div>
</div>

### Test Generalizacji: Asian Dragon $\rightarrow$ Teapot
Wizualizacja skuteczności użycia wyuczonych wag do transformacji niezależnego obiektu, jakim jest `asian_dragon`.

<div style="text-align: center;">
    <img src="output\200_0.001_4096\photos\dsfs_1.0.png" alt="Asian Dragon na modelu Armadillo" width="500"/>
    <p><i>Wektor przekształcenia Armadillo-Flow zastosowany na Asian Dragon</i></p>
</div>

## 8. Wnioski

Na podstawie przeprowadzonych testów zidentyfikowano poniższe konkluzje:

1. **Skuteczność uczenia:** Modele oparte o DGCNN z powodzeniem wyuczyły się lokalnych przemieszczeń punktów w docelowe miejsca czajnika dla znanych z treningu obiektów. Świadczą o tym bardzo niskie wartości odległości Chamfera na zbiorach treningowych (ok. 0.0026 dla modelów królika i pancernika).
2. **Jakość segmentacji / IoU:** Najwyższe dopasowanie geometryczne mierzone przez IoU oraz współczynnik Dice'a odnotowano dla przepływu `bunny_flow` (IoU: 0.2637). Przypuszczalnie uproszczona, bardziej zwarta topologia królika była łatwiejsza do adaptacji w obiekt z rączką i dzióbkiem.
3. **Wpływ augmentacji:** Dodanie mechanizmów augmentacji w postaci losowej rotacji (przy użyciu kwaternionów / macierzy z modułu `scipy.spatial.transform`) oraz drobnego losowego skalowania zapobiegło skrajnemu overfittingowi do narzuconego z góry układu globalnego, ucząc model skupienia się na relacjach geometrycznych grafu KNN.
4. **Generalizacja (Asian Dragon):** Przetestowanie modelu wektorowego wyuczonego np. na `dragon_small` względem obiektu całkowicie nowego – `asian_dragon` skutkowało wyraźnym pogorszeniem wyników. Metryki IoU i Dice znacząco spadły, a błąd Chamfera wzrósł (nawet ponad trzykrotnie dla modelu królika, 0.0087). Świadczy to o tym, że architektura uczy się w znacznym stopniu deformować konkretny typ geometrii wyjściowej i jej generalizacja na niespotykane obiekty jest ograniczona.